# 转置卷积


## 步骤
- 输入图像 $X$ 每个像素 $x$ 与卷积核 $K$ 中所有元素相乘
- 得到和卷积核大小相同的矩阵 $K'$
- 每个 $K'$ 按原始像素位置放置
- 重叠处求和得到输出图像 $Y$

![trans_conv](./images/trans_conv.svg)

### 步幅
$x$ 移动一格时，$K'$ 移动的格数就是卷积的步幅

转置卷积步幅 $\gt 1$ 时，可以增大高宽

> 也相当于把 $X$ 的像素拉开，空隙插入 $s-1$ 个零，再做普通转置卷积

### 填充
和卷积填充相反，剥离 $Y$ 外缘的 $p$ 圈像素

## 转置
对于卷积 $Y = X * K$
- 可以对 $K$ 构造一个带状矩阵 $V$
- $X, Y$ 展开为向量 $X', Y'$
- 使卷积相当于矩阵乘法 $Y' = VX'$

此时转置卷积等价于 $X' = V^TY'$
- 卷积: $(h, w) \to (h', w')$
- 转置卷积: $(h', w') \to (h, w)$

## 形状换算
输入高（宽）为 $n$，卷积核 $k$，步幅 $s$，填充 $p$，输出高（宽）为 $n'$
- 卷积: $n' = \cfrac{n + 2p - k}{s} + 1$
- 转置卷积: $n = s(n' - 1) + k - 2p$

如果让高宽成倍增加，则 $k=2p+s$


## 实现


In [1]:
import torch
from torch import nn
import util

/root/autodl-tmp/envs/daily_learning/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


基本转置卷积运算


In [2]:
def trans_conv(X, K):
    h, w = K.shape
    # 创建一个全 0 矩阵承接输出
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i:i+h, j:j+w] += X[i, j] * K # 逐位置叠加 K'
    return Y

验证


In [3]:
K = torch.FloatTensor([[1, 2], [3, 4]])
X = torch.FloatTensor([[0, 1], [2, 3]])
trans_conv(X, K)

tensor([[ 0.,  1.,  2.],
        [ 2., 10., 10.],
        [ 6., 17., 12.]])

直接调用 api 也是一样的


In [4]:
X = X.reshape((1, 1, 2, 2)) # (N, C, H, W)
K = K.reshape((1, 1, 2, 2)) # (C_out, C_in, H, W)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K
tconv(X).squeeze() # (N, C, H, W) -> (H, W)

tensor([[ 0.,  1.,  2.],
        [ 2., 10., 10.],
        [ 6., 17., 12.]], grad_fn=<SqueezeBackward0>)

填充、步幅和通道数


In [5]:
tconv = nn.ConvTranspose2d(1, 1, kernel_size=3, padding=1, bias=False) # 剥离一圈
tconv.weight.data = K
tconv(X).squeeze((0, 1)) # (N, C, H, W) -> (H, W)

tensor([[10.]], grad_fn=<SqueezeBackward2>)

In [6]:
tconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, bias=False)
tconv.weight.data = K
tconv(X).squeeze()

tensor([[ 0.,  0.,  1.,  2.],
        [ 0.,  0.,  3.,  4.],
        [ 2.,  4.,  3.,  6.],
        [ 6.,  8.,  9., 12.]], grad_fn=<SqueezeBackward0>)

In [7]:
conv = nn.Conv2d(1, 8, kernel_size=2, stride=2, padding=2)
tconv = nn.ConvTranspose2d(8, 1, kernel_size=2, stride=2, padding=2)
Y = conv(X)
X.shape, Y.shape, tconv(Y).shape

(torch.Size([1, 1, 2, 2]), torch.Size([1, 8, 3, 3]), torch.Size([1, 1, 2, 2]))

与矩阵乘法的联系


In [12]:
def conv_to_mm(X, K):
    """卷积转矩阵乘法"""
    h, w = K.shape
    V = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1, X.shape[0], X.shape[1]))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            V[i:i+h, j:j+w, i, j] = K
    return V.reshape((X.shape[0] + h - 1) * (X.shape[1] + w - 1), X.numel())
V = conv_to_mm(X.squeeze(), K.squeeze())
V

tensor([[1., 0., 0., 0.],
        [2., 1., 0., 0.],
        [0., 2., 0., 0.],
        [3., 0., 1., 0.],
        [4., 3., 2., 1.],
        [0., 4., 0., 2.],
        [0., 0., 3., 0.],
        [0., 0., 4., 3.],
        [0., 0., 0., 4.]])

In [13]:
Y = torch.matmul(V, X.squeeze().reshape(-1))
Y.reshape(3, 3)

tensor([[ 0.,  1.,  2.],
        [ 2., 10., 10.],
        [ 6., 17., 12.]])

In [14]:
Z = torch.matmul(V.T, Y.reshape(-1))
Z.reshape(2, 2)

tensor([[ 48.,  75.],
        [108., 129.]])